# Wikidataparsing

In [1]:
import json 
from rapidfuzz import fuzz
from wikidataparsing.wikidataparsing import WhatLinksHere, WikidataParser
%load_ext autoreload
%autoreload 2

import os
from utils.utils import loadWhatLinksHereLinks, loadEntitiesData, loadWikidataLinks, load_dictrel

n_core = 6


In [10]:
project = f'projects/test'
os.makedirs(project, exist_ok=True)
print(project)


projects/test


In [12]:
dict_rel = [
    {
        # specifiy by what Property must search in the WhatLinksHere pages (e.g. Q5 ('human'))
        "type": 'Q5',
        # you can provide a label for the Property (e.g. 'human')
        "name": "human",
        # indicate the set of relations you want to collect from Wikidata / Wikipedia
        "props":{
            "P19": 'placeOfBirth',
            "P569": "dateOfBirth",
            "P509": 'causeOfDeath',
            "P570": "dateOfDeath",
            "P119": "placeOfBurial",
            "P26": "spouse",
            "P106": "occupation",
            "P69": "educatedAt",
        }      
    }]

entitytype = {x['type']: x['name'] for x in dict_rel}


with open(f'{project}/dict_rel.json', 'w', encoding='utf-8') as f:
    json.dump(dict_rel, f, indent=4)

with open(f'{project}/entity_type.json', 'w', encoding='utf-8') as f:
    json.dump(entitytype, f, indent=4)


In [9]:
# entitytype = {
#     'Q5': "person",
#     'Q2221906': "location",
# }
# list_entitytype = list(entitytype.keys())
# project = '-'.join(list(entitytype.keys()))
# project = f'../data/{project}'
 
# below, example when collecting data for multiple entity types (here, person and location)
dict_rel = [
    {
        "type": 'Q5',
        "name": "person",
        "props":{
            "P19": 'placeOfBirth',
            "P569": "dateOfBirth",
            "P509": 'causeOfDeath',
            "P570": "dateOfDeath",
            "P119": "placeOfBurial",
            "P26": "spouse",
            "P106": "occupation",
            "P69": "educatedAt",
            "P509": "causeOfDeath"
        }      
    },
    {   
        "type": "Property:P625",
        "name": "location",
        'props':{
            "P571": 'inception',
            "P17": "country",
            "P1082": "population",
            "P1376": "capitalOf",
            "P276": "location",
            "P36":"capital",
            "P35": "headOfState",
            "P6": "headOfGoverment",
            "P1082": "population",
            "P47": "sharesBordersWith",
            "P463": "memberOf",
            "P206": "nextInBodyWater"
        }
    }
]

entitytype = {x['type']: x['name'] for x in dict_rel}
entitytype

{'Q5': 'person', 'Property:P625': 'location'}

## 1 - WhatLinksHere

The WhatLinksHere page on Wikidata provides a list of Wikidata pages, based on their Item id, Property id, and other parameters. In order to collect the Wikidata pages related to Items, we first collect the WhatLinksHere pages of Items having a specific Property. For instance, we collect the pages listing entities having the Q5 ("human") property.

The URLs are stored in a **ID-whatlinkshere.json** file in a **whatlinkshere** folder in the provided project folder, where ID is the property provided to search for Items (e.g. Q5). Creates a separate JSON file for each ID. 

### Collect pages

Run the cell below if you want to collect WhatLinksHere pages

In [13]:
wlh = WhatLinksHere()

In [14]:
# total of pages to collect
limit = 10
# specifies at which steps saves collected entity ids to disk
save_step = 20
# specifies how many ids to collect per pages
m_size = 50
list_urls = wlh.getWhatLinksHere(entitytype=entitytype, limit=limit, save_step=save_step, m_size=m_size, folderpath=project)
list_urls

Processing Q5 type...
Starting from last parsed url...
Done processing Q5 type


[{'type': 'Q5',
  'urls': ['https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&dir=next&offset=0%7C1069',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C1719&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C2175&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C2865&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C3995&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C4957&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C5441&dir=next',
   'https://www.wikidata.org/w/in

### Load WhatLinksHere pages

Run the cell below to load the collected URLs of WhatLinksHere pages

In [15]:
# must provide the list of entity type (e.g. Q5) to select the whatlinkshere.json file to process
list_urls = loadWhatLinksHereLinks(entity_type=entitytype, folderpath=project)
list_urls

[{'type': 'Q5',
  'urls': ['https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&dir=next&offset=0%7C1069',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C1719&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C2175&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C2865&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C3995&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C4957&dir=next',
   'https://www.wikidata.org/w/index.php?title=Special:WhatLinksHere/Q5&namespace=0&limit=100&offset=0%7C5441&dir=next',
   'https://www.wikidata.org/w/in

### Collect Item Ids

Run this cell to process the Whatlinkshere pages and collect the list of Item ids.

The results are saved in a **wikidatalinks** folder in the project folder. Item Ids are saved in a **ID-wikidatalinks.json** file, where ID is the provided Property (e.g. Q5)


In [16]:
list_entities = wlh.multi_getWikidataLinks(list_urls=list_urls, n_core=n_core, savepath=project)
list_entities

Processing Q5 type...
Processing Q5 done


[{'type': 'Q5',
  'ent_id': ['Q23',
   'Q42',
   'Q80',
   'Q1868',
   'Q2001',
   'Q35',
   'Q76',
   'Q91',
   'Q157',
   'Q181',
   'Q185',
   'Q186',
   'Q192',
   'Q206',
   'Q207',
   'Q254',
   'Q255',
   'Q260',
   'Q272',
   'Q296',
   'Q297',
   'Q301',
   'Q302',
   'Q303',
   'Q306',
   'Q307',
   'Q315',
   'Q320',
   'Q326',
   'Q329',
   'Q331',
   'Q335',
   'Q345',
   'Q346',
   'Q352',
   'Q353',
   'Q354',
   'Q360',
   'Q368',
   'Q377',
   'Q379',
   'Q392',
   'Q400',
   'Q407',
   'Q409',
   'Q410',
   'Q440',
   'Q444',
   'Q448',
   'Q449',
   'Q464',
   'Q475',
   'Q489',
   'Q493',
   'Q498',
   'Q501',
   'Q502',
   'Q504',
   'Q512',
   'Q517',
   'Q529',
   'Q530',
   'Q535',
   'Q539',
   'Q555',
   'Q557',
   'Q558',
   'Q559',
   'Q561',
   'Q562',
   'Q563',
   'Q567',
   'Q576',
   'Q579',
   'Q590',
   'Q600',
   'Q603',
   'Q605',
   'Q607',
   'Q609',
   'Q615',
   'Q619',
   'Q624',
   'Q632',
   'Q633',
   'Q635',
   'Q636',
   'Q640',
   'Q651',

Run this cell to load list of entities that have already been collected

In [17]:
list_entities = loadWikidataLinks(entity_type=entitytype, folderpath=project)
list_entities

[{'type': 'Q5',
  'ent_id': ['Q23',
   'Q42',
   'Q80',
   'Q1868',
   'Q2001',
   'Q35',
   'Q76',
   'Q91',
   'Q157',
   'Q181',
   'Q185',
   'Q186',
   'Q192',
   'Q206',
   'Q207',
   'Q254',
   'Q255',
   'Q260',
   'Q272',
   'Q296',
   'Q297',
   'Q301',
   'Q302',
   'Q303',
   'Q306',
   'Q307',
   'Q315',
   'Q320',
   'Q326',
   'Q329',
   'Q331',
   'Q335',
   'Q345',
   'Q346',
   'Q352',
   'Q353',
   'Q354',
   'Q360',
   'Q368',
   'Q377',
   'Q379',
   'Q392',
   'Q400',
   'Q407',
   'Q409',
   'Q410',
   'Q440',
   'Q444',
   'Q448',
   'Q449',
   'Q464',
   'Q475',
   'Q489',
   'Q493',
   'Q498',
   'Q501',
   'Q502',
   'Q504',
   'Q512',
   'Q517',
   'Q529',
   'Q530',
   'Q535',
   'Q539',
   'Q555',
   'Q557',
   'Q558',
   'Q559',
   'Q561',
   'Q562',
   'Q563',
   'Q567',
   'Q576',
   'Q579',
   'Q590',
   'Q600',
   'Q603',
   'Q605',
   'Q607',
   'Q609',
   'Q615',
   'Q619',
   'Q624',
   'Q632',
   'Q633',
   'Q635',
   'Q636',
   'Q640',
   'Q651',

## 2 - WikidataParser

### Parameters setting

In [20]:
# language code to select the language of the corpus
lg = 'en'
# lg = 'fr'

wp = WikidataParser(lg=lg)

# select type of Wikimedia content to process (e.g. Wikipedia, Wikinews...). Only tested with Wikipedia for now
source_doc = 'wikipedia'

# similarity threshold for the distant supervision step
score_cutoff = 95
# method to measure similarity during distant supervision step
scorer = fuzz.partial_ratio

# getOther = False
getOther = True
maxsizesent = True


In [22]:
# run this code if you want to filter the amount of entities to process 
min_limit = 0
max_limit = 100
list_entities = [{'type': x['type'], 'ent_id':x['ent_id'][min_limit:max_limit]} for x in list_entities]



In [18]:
# # loading collected data 
# list_entities_data = loadEntitiesData(savepath=project)
# len(list_entities_data), list_entities_data[0]

IndexError: list index out of range

In [19]:
# dict_rel = load_dictrel(project)
# dict_rel

FileNotFoundError: [Errno 2] No such file or directory: 'projects/test/dict_prop.json'

### Process list of entities

To use to process whole list of entities ID

In [15]:

list_entities_data = wp.processListEntities(list_entities=list_entities, dict_rel=dict_rel, scorer=scorer, score_cutoff=score_cutoff, source_doc=source_doc, getOther=getOther, maxsizesent=maxsizesent,n_core=n_core, savepath=project)

# with open(f"{project}/dict_rel.json", 'w', encoding='utf-8') as f:
#     json.dump(dict_rel, f, indent=4)


# with open(f"{project}/entity_types.json", 'w', encoding='utf-8') as f:
#     json.dump(entitytype, f, indent=4)

Step 1/6
Processing Q2221906 type...
Processing Q2221906 done
Entity data collected
Step 2/6
Content collected
Step 3/6
Entity labels collected
Step 4/6
Properties collected
Step 5/6
Sentences collected
Step 6/6
Other sentences collected


### Step by step Wikidata processing
To use in case there's a need to process that step by step

In [23]:
# retrieve JSON data for entities 

list_entities_data = wp.multi_getEntityData(list_entities=list_entities, n_core=n_core, savepath=project)



Processing Q5 type...
Processing Q5 done


In [24]:
# retrieve Wikipedia content associated with entities
list_entities_data = wp.multi_getEntityWikipediaContent(list_entities_data=list_entities_data,n_core=n_core, savepath=project)

In [25]:
# retrieve entities labels and aliases
list_entities_data = wp.multi_getEntityLabels(list_entities_data=list_entities_data,n_core=n_core, savepath=project)


In [26]:
# retrieves associated target for each selected properties
list_entities_data = wp.multi_getProperty4Entity(list_entities_data=list_entities_data, dict_rel=dict_rel, n_core=n_core, savepath=project)


In [27]:
# retrieves sentences associated with properties, e.g. distant supervision step
list_entities_data = wp.multi_findSentences(list_entities_data=list_entities_data, source_doc=source_doc, n_core=n_core, scorer=scorer, score_cutoff=score_cutoff, savepath=project)

In [28]:
# retrieve Other sentences, i.e. sentences not labelled with a property
maxsizesent = True
list_entities_data = wp.multi_getOtherSentences(list_entities_data=list_entities_data, maxsizesent=maxsizesent, n_core=n_core, savepath=project)
